# T18 - Milestone Project 3 Test

**Course section:** 18 - Milestone Project - 3

**Course notebooks covered:**
- `18-Milestone Project - 3/01-Final Capstone Project.ipynb`
- `18-Milestone Project - 3/02-Final Capstone Project Ideas.ipynb`

**Supporting course notebooks** (techniques used by the exercises below):
- `12-Advanced Python Modules/00-Collections-Module.ipynb` (`Counter`, `defaultdict`)
- `17-Advanced Python Objects and Data Structures/08-BONUS - With Statement Context Managers.ipynb` (`with open(...)`)
- `17-Advanced Python Objects and Data Structures/04-Advanced Dictionaries.ipynb` (dict comprehensions, `items()`)

**Capstone brief:** build a small program that lets the user query a movie database. Every movie record holds a `title`, a `year`, a `rating`, a list of `actors` and a `director`. Results must be saved to a local JSON file that can be reloaded later and edited.

**10 questions / 20 points** (2 coding exercises x 2, 8 design/rubric questions x 2)

**Suggested time budget:** 120 to 180 minutes.

**How the check cells work:** the two coding exercises are followed by a hidden reference solution and a cell tagged `check` that runs `assert` statements against the exact function names given in the question. Write your answer in the answer cell, then run the `check` cell; if it raises no error your answer is correct. The eight design questions are self-scored: read the reference answer in the collapsed solution block and mark your own work against it. See `S0 - Testing Syllabus.ipynb` for the full instructions, the grading command, and the submission rules.

## Q1. Coding

Write two module-level functions that handle the JSON file for the movie database:

**`load_movies(path)`**
- Takes a path to a JSON file and returns the list of movie records stored in it.
- Each movie record is a dictionary with the keys `'title'`, `'year'`, `'rating'`, `'actors'` (a list of strings) and `'director'`.
- If the file does not exist, return an empty list `[]` instead of raising an error.

**`save_movies(movies, path)`**
- Takes a list of movie dictionaries and a path, and writes the list to that path as JSON.
- Use the `json` module and the `with` statement so the file is always closed, for example `with open(path, "w") as handle: json.dump(movies, handle, indent=2)`.
- Return `None`.

The two functions must round-trip: saving a list and loading it straight back must give back an equal list.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
import json
import os


def load_movies(path):
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def save_movies(movies, path):
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(movies, handle, indent=2)
```

**Why:** `os.path.exists` guards against a missing file, and the `with` statement guarantees the file handle is closed even if writing fails. `json.load` and `json.dump` handle the conversion to and from plain Python lists and dictionaries.

</details>

In [ ]:
import os
import tempfile

folder = tempfile.mkdtemp()
path = os.path.join(folder, "movies.json")

movies = [
    {"title": "The Matrix", "year": 1999, "rating": 8.7,
     "actors": ["Keanu Reeves", "Carrie-Anne Moss"], "director": "The Wachowskis"},
    {"title": "Alien", "year": 1979, "rating": 8.5,
     "actors": ["Sigourney Weaver"], "director": "Ridley Scott"},
]

save_movies(movies, path)
assert os.path.exists(path)
assert load_movies(path) == movies
assert load_movies(os.path.join(folder, "does_not_exist.json")) == []

## Q2. Coding

Write a function **`search_movies(movies, keyword)`** that filters a list of movie dictionaries.

- `keyword` may be an empty string or `None`. In that case return a copy of the whole list (a new list, not the same object).
- Otherwise the keyword is matched **case-insensitively** against the movie's `'title'`, its `'director'` and each of its `'actors'`.
- A movie matches when the keyword appears anywhere inside any of those fields (a substring match, not an exact match).
- Return a new list of the matching movie dictionaries, preserving the original order.

A movie record always has the keys `'title'`, `'year'`, `'rating'`, `'actors'` and `'director'`.

In [ ]:
# your answer here

<details>
<summary>Solution</summary>

```python
def search_movies(movies, keyword):
    if not keyword:
        return list(movies)

    needle = keyword.lower()
    matches = []
    for movie in movies:
        haystack = [movie["title"], movie["director"]] + list(movie["actors"])
        if any(needle in field.lower() for field in haystack):
            matches.append(movie)
    return matches
```

**Why:** Lower-casing both sides makes the comparison case-insensitive, and building one list of searchable fields lets a single `any` call cover title, director and actors. Returning `list(movies)` for an empty keyword still gives the caller a fresh list it can safely modify.

</details>

In [ ]:
movies = [
    {"title": "The Matrix", "year": 1999, "rating": 8.7,
     "actors": ["Keanu Reeves", "Carrie-Anne Moss"], "director": "The Wachowskis"},
    {"title": "Alien", "year": 1979, "rating": 8.5,
     "actors": ["Sigourney Weaver"], "director": "Ridley Scott"},
    {"title": "Arrival", "year": 2016, "rating": 7.9,
     "actors": ["Amy Adams", "Jeremy Renner"], "director": "Denis Villeneuve"},
]

assert search_movies(movies, "matrix") == [movies[0]]
assert search_movies(movies, "SIGOURNEY") == [movies[1]]
assert search_movies(movies, "renner") == [movies[2]]
assert search_movies(movies, "villeneuve") == [movies[2]]
assert search_movies(movies, "not in the database") == []
assert search_movies(movies, "") == movies and search_movies(movies, "") is not movies

## Q3. Design

Your program stores movies. Should each movie be a **plain dictionary** or a **custom class** (for example `class Movie:` with `__init__`)?

Describe both options and give your recommendation, with at least one concrete reason from each side.

<details>
<summary>Solution</summary>

**Reference answer**

*Plain dictionary:* cheap to build, prints and compares nicely, and it is exactly what `json.dump` / `json.load` produce, so saving and reloading a list of dictionaries needs no conversion code at all. Its downside is that keys are looked up by string at runtime, a typo raises a `KeyError` only when that line runs, and every movie must have the same keys written out by hand.

*Custom class:* the data model is explicit, attributes can have defaults, and you can attach behaviour such as `Movie.average_rating()` or a `__str__` that formats the record for the user. Its downside is that it does not serialise directly, so you need `to_dict()` / `from_dict()` conversion code around `json.dump` / `json.load`, which is extra code to keep in sync.

**Recommendation:** start with plain dictionaries, because JSON is the storage format for this project and dictionaries keep the save/load code trivial. If the program grows and the same field-manipulation logic is repeated everywhere, refactor to a class and add the two conversion methods. Either way, keep the shape of a record (the five keys) in one place, and document it.

</details>

## Q4. Design

Describe how your program **loads** the database at start-up and **saves** it back after a change.

Your answer must mention: the file format, the module you use, whether you load once or on every query, and how you make sure the file is closed properly.

<details>
<summary>Solution</summary>

**Reference answer**

*File format:* JSON, using the `json` module from the standard library. It is plain text, human-readable with `indent=2`, and it maps directly onto Python lists and dictionaries.

*Loading:* read the whole file once at start-up into an in-memory list of movie dictionaries (`movies = load_movies(DB_PATH)`), and use that list for every query during the session. Reading the file on every single query is wasteful and slow, and it also makes it easy to lose unsaved edits.

*Saving:* write the whole list back after every change the user makes (`save_movies(movies, DB_PATH)`), so the file on disk is always up to date and a crash cannot lose work. For a larger collection you would save on exit or on a timer instead.

*Closing the file:* use the `with` statement, `with open(path, "w") as handle: json.dump(movies, handle, indent=2)`. The `with` block closes the handle on the way out, including when an exception is raised inside the block, which is exactly the behaviour taught in the *With Statement Context Managers* bonus notebook.

</details>

## Q5. Design

Your program must let the user query the database in three ways: **by title**, **by year** and **by rating**.

Sketch the functions you would write for each query and say what each one returns. What should happen when a query matches nothing?

<details>
<summary>Solution</summary>

**Reference answer**

Three small, focused functions, each taking the movie list and returning a new list:

- `search_by_title(movies, title)` - case-insensitive substring match on the `'title'` field, returning the matching movies in their original order.
- `search_by_year(movies, year)` - exact match on the `'year'` field (you could also offer a range such as `year_from` / `year_to`); returning a list keeps the interface identical to the other two queries.
- `search_by_rating(movies, minimum)` - returns every movie whose `'rating'` is greater than or equal to `minimum`, sorted from highest rating to lowest.

Every query function returns a list, possibly empty, and never modifies the original list. The caller prints a friendly message such as `No movies matched that search.` when the returned list is empty, and otherwise prints each movie's title, year, rating, actors and director. Separating "find the matches" from "display the matches" keeps the query functions testable and lets you reuse them in the GUI project.

</details>

## Q6. Design

What should your program do when the JSON file is **missing** or **corrupt** (for example it was edited by hand and is no longer valid JSON)?

Describe the behaviour you would implement, and say where in the code that behaviour belongs.

<details>
<summary>Solution</summary>

**Reference answer**

*Missing file:* this is a normal situation on the first run, not an error. `load_movies` should check `os.path.exists(path)` and return an empty list when the file is absent. The program then simply starts with an empty database, and the first `save_movies` call creates the file.

*Corrupt file:* this is a real error, but the program should still not crash with a raw traceback. Wrap the `json.load` call in a `try` / `except json.JSONDecodeError` block, print a clear message such as `The database file is not valid JSON, starting with an empty database.`, and return an empty list. A more defensive version also checks the top-level type, because valid JSON can still be the wrong shape (a list of numbers instead of a list of movie dictionaries), and skips or reports records that are missing the expected keys.

*Where the behaviour belongs:* in the loading function, not in the user interface code. The loader is the single place that knows about the file format, so every caller gets the same safe behaviour, and the UI never has to think about missing or broken files.

</details>

## Q7. Design

Later you want to add a new field to every movie, for example `'genres'` (a list of strings) or `'runtime'` (an integer number of minutes).

Describe **how** you would extend the program, and explain what happens to data that was saved before the change.

<details>
<summary>Solution</summary>

**Reference answer**

*Changing the code:* add the new key to every place a record is created (the "add a movie" function, the sample/seed data, and any place that builds a movie dictionary). If you use plain dictionaries, the change is small because you can add a key to an existing dictionary with `movie["genres"] = []`, and you can use `setdefault` to give a default value when reading a record that predates the change: `movie.setdefault("genres", [])`.

*Old data:* a dictionary loaded from an older JSON file simply has no `'genres'` key, so reading `movie["genres"]` raises a `KeyError`. The fix is either to read it defensively with `movie.get("genres", [])` (or `setdefault`), or to write a one-off migration that loads the old file, adds the missing key to each record with a sensible default, and saves it back. A migration step is the cleaner option because it fixes the data once instead of scattering `.get` calls through the whole program.

*Design habit:* keep the record shape in a single place, such as a `FIELDS` list or a small factory function `make_movie(...)`, so that adding a field means editing one place and then running the migration over the existing file.

</details>

## Q8. Design

Your program should let the user **add a rating** to a movie.

Explain two different ways you could store ratings, and say what each one lets you do afterwards.

<details>
<summary>Solution</summary>

**Reference answer**

*Option 1 - one rating per movie:* keep a single `'rating'` number on the record and overwrite it when the user adds a new one. Simple, and the record stays small. But the history is lost, and you cannot compute an average over several opinions or show how a movie's score changed over time.

*Option 2 - a list of ratings:* store every rating in a list, for example `'ratings': [8.0, 9.0, 7.5]`, and expose the current score as the average of that list. Now you can show how many people rated the movie, compute a mean, and keep the history. The trade-off is a little more code (you must update the average whenever the list changes) and a slightly bigger file.

A good middle ground for a small project is to keep the single `'rating'` field as the displayed value and, if you want history, keep the list as an extra key and recompute the average when saving.

Whichever you choose, validate the input first: reject a rating that is not a number, and clamp or reject values outside 0 to 10.

</details>

## Q9. Design

Describe how you would **structure the code** so that the parts can be reused and tested separately.

Your answer should mention at least: how you separate the data handling from the user interaction, and where the "start here" code lives.

<details>
<summary>Solution</summary>

**Reference answer**

*Separate the data layer from the interaction layer.* The data layer is a set of small, pure-ish functions that take data and return data: `load_movies`, `save_movies`, `search_movies`, `search_by_year`, `search_by_rating`, `add_movie`, `add_rating`. They never call `input()` and never `print()`; they take arguments and return values. The interaction layer is a menu loop that calls `input()`, passes the answers into the data layer and prints the results. Because the data layer never talks to the user, you can test it without any input, and you can reuse it later from the GUI project or a notebook.

*Keep the "start here" code at the bottom.* Put the entry point in a `if __name__ == "__main__":` block that calls a `main()` function. Importing your module from another script (or from a test file) then has no side effects, because the menu only runs when the file is executed directly.

*Other habits that help:* one function per task, descriptive names, docstrings, and constants such as `DB_PATH = "movies.json"` at the top instead of repeating string literals. A menu-driven program is a natural fit here because the milestone asks for a program the user can drive from the terminal.

</details>

## Q10. Design

Describe **how you would test** the program.

Your answer should cover: which parts are worth testing automatically, what a good test case looks like, and why you should not use the real `movies.json` file while testing.

<details>
<summary>Solution</summary>

**Reference answer**

*What is worth testing automatically:* the data-handling functions. They take plain arguments and return plain values, so they are trivial to test: `save_movies` then `load_movies` must return an equal list; `load_movies` on a path that does not exist must return `[]`; `search_movies` must match on title, on actor and on director, must be case-insensitive, and must return `[]` for a keyword that matches nothing.

*What a good test case looks like:* a small, fixed list of two or three movie dictionaries that you build inside the test, plus an assertion for one specific behaviour at a time. Fixed sample data makes the expected results obvious, and one assertion per behaviour tells you exactly what broke when a test fails. The `assert` statement is the natural tool here: it raises `AssertionError` with a message you can read.

*Why not the real file:* tests must not depend on or damage real data. Point the functions at a temporary path (for example with `tempfile.mkdtemp()`) so each run writes to a throwaway folder, and never write into the project folder itself. Tests should also be repeatable and not depend on the current date, the network, or random values.

*What to test manually:* the menu flow and the printed output. Those are easier to check by running the program and looking at the screen, so keep the testing effort where it pays off.

</details>

## Topics covered in this test

- **Capstone scope:** the Final Capstone Project brief and the project-ideas list, applied to a movie database with `title`, `year`, `rating`, `actors` and `director`.
- **Data model:** plain dictionaries versus a custom `Movie` class, and how each one serialises to JSON.
- **Persistence:** the `json` module (`json.load`, `json.dump`), reading and writing a local JSON file, and the `with open(...)` context manager for safe file handling.
- **Querying:** case-insensitive substring search over title, director and actors, plus querying by year and by minimum rating, and returning an empty list when nothing matches.
- **Robustness:** missing files, corrupt JSON, and the wrong top-level JSON shape.
- **Extending the program:** adding new fields, defaults with `.get` / `setdefault`, and migrating an existing file.
- **Ratings:** a single rating per movie versus a list of ratings with an average.
- **Code structure:** a data layer separated from the interaction layer, a `main()` entry point guarded by `if __name__ == "__main__":`, and reusable query functions.
- **Testing:** assertions on the data functions, fixed sample data, and temporary files so the real database is never touched.